# Notebook 1 — KiTS23 AML Augmentation Pipeline
**Dataset:** KiTS23 | **Format:** NIfTI (.nii.gz) | **Label:** AML only

---

## What This Notebook Does

This notebook performs image-level data augmentation on AML-labeled cases from the KiTS23 dataset. Three augmented versions are created per case — eroded mask, dilated mask, and flipped image+mask. Augmented files are saved as new NIfTI files and treated as independent new images in Notebook 3 (KiTS23 feature extraction).

**Pipeline position:**
```
[Notebook 1 — YOU ARE HERE] → [Notebook 2] → [Notebook 3] → [Notebook 4]
 KiTS23 Augmentation           UCSF Augment   KiTS23 Extr   UCSF Extr
```

---

## Why Image-Level Augmentation Before Feature Extraction

The dataset has a ~9:1 RCC:AML imbalance. AML is the minority class. Standard oversampling methods like SMOTE operate on already-extracted CSV features and treat them as independent variables. Radiomics features are mathematically coupled — Energy, Entropy, and Kurtosis all derive from the same intensity histogram. SMOTE interpolation between feature rows can create physically impossible combinations that do not correspond to real tissue.

By augmenting at the image level first and then extracting features, all features remain internally consistent. PyRadiomics computes each augmented sample's full feature set from its own image — the biological relationships between features are preserved by construction.

**Reference:** Bianchini et al. (2024), Journal of Digital Imaging
https://link.springer.com/article/10.1007/s10278-024-01013-0
Perturbation-based augmentation outperformed SMOTE and ADASYN: F1 80-86%, AUC 0.91-0.92.

---

## Augmentation Methods

| Method | Applied To | Factor | Biological Rationale |
|--------|-----------|--------|---------------------|
| Erosion | Mask only | 30% volume reduction | Simulates conservative segmentation by a radiologist |
| Dilation | Mask only | 30% volume increase | Simulates generous segmentation by a radiologist |
| Flip | Image + Mask | axis=0 | Mirror patient — valid because kidneys are anatomically symmetric |

**Erosion and Dilation — mask only:** These are ROI perturbations, not geometric transformations. The CT Hounsfield Unit values are completely untouched. Only the tumor boundary changes, simulating inter-observer segmentation variability. Features are still extracted from original, unmodified image intensities — just from a slightly different region. The original image is copied unchanged alongside the perturbed mask.

**Flip — both image and mask:** Flipping is a geometric transformation. If only the mask were flipped, it would describe a tumor region on the wrong anatomical location — for example the mask would outline the right kidney while the CT image shows the left kidney. This would teach the model that healthy tissue is tumor. Both must be flipped together to preserve the spatial relationship between tumor texture and surrounding anatomy.

---

## Key Fix vs. Original Pipeline: Affine Matrix After Flip

The original pipeline called `np.flip(image_data, axis=0)` and saved the result with the original unchanged affine matrix. The affine encodes the spatial orientation of the volume in real-world coordinates. After flipping, the voxel array is reversed in memory but the affine still describes the original orientation — a spatial inconsistency.

PyRadiomics uses the affine to determine voxel directions when computing directional texture features (GLRLM run directions, GLCM angular offsets). An incorrect affine after flipping causes these features to be computed with the wrong orientation reference.

**Fix:** After `np.flip`, the affine column corresponding to the flip axis is negated and the origin is shifted to the new starting corner of the volume. This is applied in `run_flip_task()` below.

---

## Output Directory Structure
```
kidney-Augmentation/
  eroded/
    case_XXXXX/imaging.nii.gz       <- copied from original (unchanged)
    case_XXXXX/segmentation.nii.gz  <- eroded mask
  dilated/
    case_XXXXX/imaging.nii.gz       <- copied from original (unchanged)
    case_XXXXX/segmentation.nii.gz  <- dilated mask
  flipped/
    case_XXXXX/imaging.nii.gz       <- flipped image with corrected affine
    case_XXXXX/segmentation.nii.gz  <- flipped mask with corrected affine
```

## 1. Install Dependencies

In [ ]:
!pip install nibabel scipy -q

## 2. Imports

In [ ]:
import os
import shutil
import numpy as np
import nibabel as nib
from scipy import ndimage

print("Libraries loaded.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths and Configuration

In [ ]:
SOURCE_DIR          = "/content/drive/MyDrive/kidney-aml/aml/"
OUTPUT_DIR          = "/content/drive/MyDrive/kidney-Augmentation"
PERTURBATION_FACTOR = 0.30   # 30% boundary change, per reference paper

cases = sorted([c for c in os.listdir(SOURCE_DIR)
                if os.path.isdir(os.path.join(SOURCE_DIR, c))])
print(f"Found {len(cases)} AML cases:")
for c in cases:
    print(f"  {c}")

## 5. Core Augmentation Functions

In [ ]:
def get_3d_erosion(mask, factor=0.30):
    """
    Iteratively erodes the 3D mask until the target volume reduction is met.
    Applied to mask only. Image is copied unchanged.
    Simulates a radiologist drawing a smaller, more conservative tumor boundary.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 - factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)  # 6-connectivity 3x3x3

    while np.sum(perturbed) > target_area:
        prev_mask = np.copy(perturbed)
        perturbed = ndimage.binary_erosion(perturbed, structure=struct)
        if np.sum(perturbed) == 0:
            print("    Warning: erosion would eliminate mask entirely. Stopping early.")
            return prev_mask
    return perturbed


def get_3d_dilation(mask, factor=0.30):
    """
    Iteratively dilates the 3D mask until the target volume growth is met.
    Applied to mask only. Image is copied unchanged.
    Simulates a radiologist drawing a larger, more generous tumor boundary.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 + factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)

    while np.sum(perturbed) < target_area:
        perturbed = ndimage.binary_dilation(perturbed, structure=struct)
    return perturbed


def run_flip_task(image_data, mask_data, affine, axis=0):
    """
    Flips both image and mask along the specified axis.
    Returns flipped arrays AND a corrected affine matrix.

    FIX vs original pipeline:
    The original code saved flipped data with the original affine, creating
    a spatial inconsistency. PyRadiomics uses the affine for directional
    texture features (GLRLM, GLCM). This function negates the flip axis
    direction and shifts the origin to correctly describe the new orientation.
    """
    flipped_image = np.flip(image_data, axis=axis)
    flipped_mask  = np.flip(mask_data,  axis=axis)

    # Correct the affine to match the flipped voxel layout
    n_voxels       = image_data.shape[axis]
    flipped_affine = affine.copy()
    flipped_affine[:, axis] = -affine[:, axis]                               # negate direction
    flipped_affine[:3, 3]   = affine[:3, 3] + (n_voxels - 1) * affine[:3, axis]  # shift origin

    return flipped_image, flipped_mask, flipped_affine


def save_medical_volume(data, affine, header, output_path):
    """Saves a numpy array as a NIfTI file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    nib.save(nib.Nifti1Image(data.astype(np.float32), affine, header), output_path)
    print(f"  Saved: {output_path}")

## 6. Main Pipeline Controller

In [ ]:
def execute_augmentation_pipeline(source_dir, root_output_dir, factor=0.30):
    """
    Iterates over all AML NIfTI cases and produces three augmented versions:
      - eroded mask   (image copied unchanged)
      - dilated mask  (image copied unchanged)
      - flipped image + mask (both with corrected affine)
    Skips cases with missing files or empty masks.
    """
    os.makedirs(root_output_dir, exist_ok=True)
    cases   = sorted([c for c in os.listdir(source_dir)
                      if os.path.isdir(os.path.join(source_dir, c))])
    skipped = []

    print(f"Starting augmentation for {len(cases)} cases...\n")

    for case in cases:
        img_p = os.path.join(source_dir, case, "imaging.nii.gz")
        msk_p = os.path.join(source_dir, case, "segmentation.nii.gz")

        if not os.path.exists(img_p) or not os.path.exists(msk_p):
            print(f"SKIPPED {case}: imaging or segmentation file missing.")
            skipped.append((case, "missing file"))
            continue

        img_obj  = nib.load(img_p)
        img_data = img_obj.get_fdata()
        msk_data = nib.load(msk_p).get_fdata()
        affine   = img_obj.affine
        header   = img_obj.header

        if np.sum(msk_data) == 0:
            print(f"SKIPPED {case}: mask is empty.")
            skipped.append((case, "empty mask"))
            continue

        print(f"Processing: {case}")

        # --- Task A: Erosion (mask only, image copied) ---
        eroded = get_3d_erosion(msk_data, factor)
        save_medical_volume(eroded, affine, header,
                            f"{root_output_dir}/eroded/{case}/segmentation.nii.gz")
        shutil.copy(img_p, f"{root_output_dir}/eroded/{case}/imaging.nii.gz")
        print(f"  Copied original image  → eroded/{case}/imaging.nii.gz")

        # --- Task B: Dilation (mask only, image copied) ---
        dilated = get_3d_dilation(msk_data, factor)
        save_medical_volume(dilated, affine, header,
                            f"{root_output_dir}/dilated/{case}/segmentation.nii.gz")
        shutil.copy(img_p, f"{root_output_dir}/dilated/{case}/imaging.nii.gz")
        print(f"  Copied original image  → dilated/{case}/imaging.nii.gz")

        # --- Task C: Flip (image AND mask, corrected affine) ---
        f_img, f_msk, f_affine = run_flip_task(img_data, msk_data, affine, axis=0)
        save_medical_volume(f_img, f_affine, header,
                            f"{root_output_dir}/flipped/{case}/imaging.nii.gz")
        save_medical_volume(f_msk, f_affine, header,
                            f"{root_output_dir}/flipped/{case}/segmentation.nii.gz")

        print(f"  Done: {case}\n")

    print("=" * 50)
    print(f"Augmentation complete.")
    print(f"  Processed : {len(cases) - len(skipped)}")
    print(f"  Skipped   : {len(skipped)}")
    if skipped:
        for s, r in skipped:
            print(f"    {s}: {r}")

## 7. Run

In [ ]:
execute_augmentation_pipeline(
    source_dir      = SOURCE_DIR,
    root_output_dir = OUTPUT_DIR,
    factor          = PERTURBATION_FACTOR
)

## 8. Verify Output and Affine Correction

In [ ]:
# File count per augmentation type
for aug in ["eroded", "dilated", "flipped"]:
    aug_dir = os.path.join(OUTPUT_DIR, aug)
    n = len(os.listdir(aug_dir)) if os.path.exists(aug_dir) else 0
    print(f"{aug}/: {n} cases")

# Confirm affine was corrected for first flipped case
print("\n--- Affine correction verification ---")
first_case = sorted(os.listdir(os.path.join(OUTPUT_DIR, "flipped")))[0]
orig_aff = nib.load(os.path.join(SOURCE_DIR,        first_case, "imaging.nii.gz")).affine
flip_aff = nib.load(os.path.join(OUTPUT_DIR, "flipped", first_case, "imaging.nii.gz")).affine
print(f"Case          : {first_case}")
print(f"Original col 0: {orig_aff[:3, 0]}")
print(f"Flipped  col 0: {flip_aff[:3, 0]}")
print(f"Correctly negated: {np.allclose(flip_aff[:3, 0], -orig_aff[:3, 0])}")